# Transporte Público · RED Movilidad — Posiciones GPS Diarias

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Observatory/datopia-notebooks/blob/main/transporte/red-movilidad/demo.ipynb)
[![Licencia](https://img.shields.io/badge/licencia-MIT-blue.svg)](../../LICENSE)
[![Dataset](https://img.shields.io/badge/dataset-RED%20Movilidad-green.svg)]()
[![Python](https://img.shields.io/badge/python-3.10%2B-blue.svg)]()

---

## Descripción

Acceso al dataset de posiciones GPS diarias de la **RED Movilidad** (Región Metropolitana, Chile)
a través del **Datopia Lakehouse**. Los datos provienen de la API DTP Metropolitano y se
consolidan diariamente en formato **GeoParquet** particionado por fecha.

El acceso es directo vía S3 con credenciales temporales, sin descarga de archivos.

### Contenido

1. Configuración y autenticación
2. Exploración del dataset (esquema, cobertura temporal)
3. Consulta semanal: registros, vehículos y servicios por día
4. Velocidad promedio por hora del día
5. Visualizaciones

### Requisitos

Cuenta en el Datopia Lakehouse · Python 3.10+ · Las dependencias se instalan automáticamente

In [ ]:
# @title Instalación de dependencias
import importlib, subprocess, sys

for paquete in ["requests", "duckdb", "pandas", "matplotlib"]:
    if importlib.util.find_spec(paquete) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

import json, os, getpass, pathlib
import requests, duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("Dependencias listas.")

In [ ]:
# @title Configuración de credenciales
EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}")

URL_API = "https://ee98cnfz7e.execute-api.us-west-2.amazonaws.com/prod"

if EN_COLAB:
    EMAIL = input("Email: ").strip()
    PASSWORD = getpass.getpass("Contraseña: ")
else:
    ruta_cfg = pathlib.Path("../../config.json")
    if not ruta_cfg.exists():
        raise FileNotFoundError(
            f"Archivo de configuración no encontrado en {ruta_cfg.resolve()}.\n"
            "Copia config.example.json → config.json y completa tus credenciales."
        )
    cfg = json.loads(ruta_cfg.read_text())
    EMAIL = cfg.get("test_user", {}).get("email") or input("Email: ").strip()
    PASSWORD = cfg.get("test_user", {}).get("password") or getpass.getpass(
        "Contraseña: "
    )

print(f"API    : {URL_API}")
print(f"Usuario: {EMAIL}")

---
## 1. Autenticación y conexión a S3

In [ ]:
# Iniciar sesión y obtener credenciales temporales S3
resp_login = requests.post(
    f"{URL_API}/auth/login",
    json={"email": EMAIL, "password": PASSWORD},
    timeout=30,
)
resp_login.raise_for_status()
token = resp_login.json()["id_token"]

resp_s3 = requests.post(
    f"{URL_API}/auth/session/s3",
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)
resp_s3.raise_for_status()
creds = resp_s3.json()

print("Sesión iniciada")
print(f"  Bucket : {creds['bucket']}")
print(f"  Región : {creds['region']}")
print(f"  Expira : {creds['expires_at']}")

In [ ]:
# Configurar DuckDB con credenciales S3
con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET s3 (
        TYPE          S3,
        KEY_ID        '{creds["access_key_id"]}',
        SECRET        '{creds["secret_access_key"]}',
        SESSION_TOKEN '{creds["session_token"]}',
        REGION        '{creds["region"]}'
    )
""")

BASE = (
    f"s3://{creds['bucket']}/categoria=transporte/pais=cl"
    f"/fuente=red-movilidad/tipo=posicion-diaria/version=v1"
)
print("DuckDB conectado a S3.")

---
## 2. Exploración del dataset

In [ ]:
# Metadatos del dataset (dataset.json en la raíz del path S3)
meta = con.execute(f"SELECT * FROM read_json('{BASE}/dataset.json')").df().iloc[0].to_dict()

print(meta["description"])
print()
print(f"Formato      : {meta['format']}")
print(f"Particiones  : {', '.join(meta['partition_keys'])}")
print(f"Geometría    : {meta['geometry']['geometry_type']} · {meta['geometry']['crs']} · {meta['geometry']['encoding']}")
print()

cols = pd.DataFrame(meta["columns"])[["name", "type", "comment"]]
cols

In [ ]:
# Esquema de columnas
con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{BASE}/year=2026/month=05/**/*.parquet') LIMIT 0
""").df()[["column_name", "column_type", "null"]]

In [ ]:
# Cobertura temporal disponible — lista archivos del lakehouse (no lee datos)
cobertura = con.execute(f"""
    SELECT
        regexp_extract(file, 'year=(\\d+)', 1)::INTEGER  AS año,
        regexp_extract(file, 'month=(\\d+)', 1)::INTEGER AS mes,
        COUNT(DISTINCT regexp_extract(file, 'day=(\\d+)', 1)::INTEGER) AS dias_disponibles
    FROM glob('{BASE}/**/*.parquet')
    GROUP BY año, mes
    ORDER BY año, mes
""").df()
print(f"Meses en el lakehouse: {len(cobertura)}")
cobertura

---
## 3. Consulta semanal

In [ ]:
# Ajusta año, mes y rango de días según la cobertura disponible arriba
AÑO = 2026
MES = 5
DIA_INICIO = 1
DIA_FIN = 7  # demo de una semana (~50-70 MB vs ~300 MB del mes completo)

# Rutas explícitas por día — DuckDB solo accede a esos 7 directorios
rutas_semana = [
    f"'{BASE}/year={AÑO}/month={MES:02d}/day={d:02d}/**/*.parquet'"
    for d in range(DIA_INICIO, DIA_FIN + 1)
]
ruta_semana = f"[{', '.join(rutas_semana)}]"

resumen_diario = con.execute(f"""
    SELECT
        day                          AS dia,
        COUNT(*)                     AS registros,
        COUNT(DISTINCT plate)        AS vehiculos,
        COUNT(DISTINCT service_name) AS servicios
    FROM read_parquet({ruta_semana}, hive_partitioning=true)
    GROUP BY dia
    ORDER BY dia
""").df()

print(
    f"Período: {AÑO}-{MES:02d} días {DIA_INICIO}–{DIA_FIN} — {len(resumen_diario)} días cargados"
)
resumen_diario

In [ ]:
# Muestra de registros GPS
con.execute(f"""
    SELECT timestamp_gps_utc, plate, service_name, speed, direction, way
    FROM read_parquet({ruta_semana}, hive_partitioning=true)
    LIMIT 10
""").df()

---
## 4. Velocidad promedio por hora del día

In [ ]:
velocidad_hora = con.execute(f"""
    SELECT
        EXTRACT(HOUR FROM timestamp_gps_utc)::INTEGER AS hora,
        ROUND(AVG(speed), 2)     AS velocidad_promedio_kmh,
        ROUND(MEDIAN(speed), 2)  AS velocidad_mediana_kmh,
        COUNT(*)                 AS registros
    FROM read_parquet({ruta_semana}, hive_partitioning=true)
    WHERE speed > 0
    GROUP BY hora
    ORDER BY hora
""").df()
velocidad_hora

---
## 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    f"RED Movilidad — {AÑO}-{MES:02d} · días {DIA_INICIO}–{DIA_FIN}",
    fontsize=16,
    fontweight="bold",
    y=1.01,
)

dias = resumen_diario["dia"].astype(str)

# Vehículos únicos por día
ax = axes[0, 0]
ax.bar(dias, resumen_diario["vehiculos"], color="#2196F3", edgecolor="white")
ax.set_title("Vehículos únicos por día", fontweight="bold")
ax.set_xlabel("Día")
ax.set_ylabel("Vehículos")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.tick_params(axis="x", rotation=45)

# Registros GPS por día
ax = axes[0, 1]
ax.bar(dias, resumen_diario["registros"] / 1e6, color="#4CAF50", edgecolor="white")
ax.set_title("Registros GPS por día", fontweight="bold")
ax.set_xlabel("Día")
ax.set_ylabel("Millones de registros")
ax.tick_params(axis="x", rotation=45)

# Velocidad por hora
ax = axes[1, 0]
ax.plot(
    velocidad_hora["hora"],
    velocidad_hora["velocidad_promedio_kmh"],
    marker="o",
    color="#FF5722",
    label="Promedio",
    linewidth=2,
)
ax.plot(
    velocidad_hora["hora"],
    velocidad_hora["velocidad_mediana_kmh"],
    marker="s",
    color="#9C27B0",
    label="Mediana",
    linewidth=2,
    linestyle="--",
)
ax.set_title("Velocidad por hora del día", fontweight="bold")
ax.set_xlabel("Hora (UTC)")
ax.set_ylabel("Velocidad (km/h)")
ax.legend()
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.3)

# Servicios activos por día
ax = axes[1, 1]
ax.bar(dias, resumen_diario["servicios"], color="#FF9800", edgecolor="white")
ax.set_title("Servicios activos por día", fontweight="bold")
ax.set_xlabel("Día")
ax.set_ylabel("Servicios")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

---
## Próximos pasos

- **Análisis espacial:** usar la columna `geometry` (WKB Point EPSG:4326) con `geopandas` para mapas de trayectorias.
- **Múltiples meses:** ajustar `AÑO`/`MES` o iterar sobre rangos para análisis temporales largos.
- **Por servicio:** filtrar por `service_name` para analizar una línea específica (ej. `T201`, `B01`).
- **Más datasets:** explorar otros conjuntos disponibles en [datopia-notebooks](https://github.com/Data-Observatory/datopia-notebooks).

---

*Datos provistos por [Data Observatory](https://dataobservatory.net) · Fuente: DTP Metropolitano (DTPM)*